# 4 - Guide d'intégration des modèles de différents packages

## Importation des modules

In [1]:
# Importation des modules
# Modules de base
import numpy as np
import pandas as pd
import sys

# Ajout du chemin
sys.path.append('..')

# Importation des utilitaires sklearn
from sklearn.utils import _safe_indexing
from sklearn.utils.metaestimators import _safe_split
from sklearn.model_selection import cross_val_predict

# Importation des modèles
# Sklearn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
# XGBoost

# Sktime
from sktime.forecasting.arima import ARIMA
# Tslearn

# Darts


# Importation des pipelines
# Sklearn
from sklearn.pipeline import Pipeline

#  Eléments du package à intégrer
# Crossval
from tsforecast.crossvals import (
    PanelOutOfSampleSplit
)

## 1. Création de données synthétiques

In [2]:
# Fonction de génération des données de panel
def generate_panel_data(entities=['A', 'B', 'C'], start_date='2015-01-01', periods=120, 
                        freq='MS', heterogeneous_effects=True, common_trend=True, 
                        entity_specific_seasonality=True, cross_sectional_correlation=0.3,
                        missing_data_prob=0.0):
    """
    Generate synthetic panel data with various characteristics.
    
    Args:
        entities: List of entity identifiers
        start_date: Start date for the panel
        periods: Number of time periods per entity
        freq: Frequency of observations
        heterogeneous_effects: Whether entities have different baseline levels
        common_trend: Whether to include a common trend across entities
        entity_specific_seasonality: Whether seasonality patterns differ by entity
        cross_sectional_correlation: Correlation between entity shocks
        missing_data_prob: Probability of missing observations
    
    Returns:
        pd.DataFrame: Panel data with MultiIndex (entity, date)
    """
    # Création de l'index temporel
    dates = pd.date_range(start=start_date, periods=periods, freq=freq)
    
    # Création du MultiIndex (entity, date)
    index = pd.MultiIndex.from_product([entities, dates], names=['entity', 'date'])
    
    # Initialisation du DataFrame
    panel_data = pd.DataFrame(index=index)
    
    # Génération des effets fixes par entité (hétérogénéité)
    if heterogeneous_effects:
        entity_effects = {entity: np.random.normal(0, 2) for entity in entities}
    else:
        entity_effects = {entity: 0 for entity in entities}
    
    # Tendance commune
    if common_trend:
        common_trend_values = 0.02 * np.arange(periods)
    else:
        common_trend_values = np.zeros(periods)
    
    # Génération de chocs corrélés entre entités
    if cross_sectional_correlation > 0:
        # Chocs communs
        common_shocks = np.random.normal(0, 1, periods)
        # Chocs idiosyncratiques
        idiosyncratic_shocks = {
            entity: np.random.normal(0, 1, periods) 
            for entity in entities
        }
    
    # Construction des séries pour chaque entité
    values = []
    
    # Parcours des entités
    for entity in entities:
        # Effet fixe de l'entité
        entity_effect = entity_effects[entity]
        
        # Saisonnalité spécifique à l'entité
        if entity_specific_seasonality:
            # Période et amplitude différentes selon l'entité
            seasonal_period = 20 + hash(entity) % 40  # Entre 20 et 60
            seasonal_amplitude = 0.5 + (hash(entity) % 100) / 200  # Entre 0.5 et 1.0
        else:
            seasonal_period = 30
            seasonal_amplitude = 0.5
        
        seasonal_values = seasonal_amplitude * np.sin(2 * np.pi * np.arange(periods) / seasonal_period)
        
        # Processus autorégressif spécifique à l'entité
        ar_coef = 0.5 + (hash(entity) % 50) / 100  # Entre 0.5 et 1.0
        ar_process = np.zeros(periods)
        ar_process[0] = np.random.normal(0, 0.5)
        for t in range(1, periods):
            ar_process[t] = ar_coef * ar_process[t-1] + np.random.normal(0, 0.5)
        
        # Combinaison des composantes
        if cross_sectional_correlation > 0:
            # Chocs avec corrélation croisée
            correlated_shocks = (
                np.sqrt(cross_sectional_correlation) * common_shocks +
                np.sqrt(1 - cross_sectional_correlation) * idiosyncratic_shocks[entity]
            )
        else:
            correlated_shocks = np.random.normal(0, 1, periods)
        
        # Combinaison des valeurs
        entity_values = (
            entity_effect + 
            common_trend_values + 
            seasonal_values + 
            ar_process + 
            correlated_shocks
        )
        
        # Ajout de données manquantes
        if missing_data_prob > 0:
            missing_mask = np.random.random(periods) < missing_data_prob
            entity_values[missing_mask] = np.nan
        
        values.extend(entity_values)
    
    # Création du DataFrame final avec les valeurs
    panel_data['value'] = values
    
    # Ajout de variables explicatives
    panel_data['lag_value'] = panel_data.groupby('entity')['value'].shift(1)
    panel_data['trend'] = np.tile(np.arange(periods), len(entities))
    panel_data['month'] = panel_data.index.get_level_values('date').month
    
    return panel_data

In [3]:
# Génération de différents types de données de panel
print("📊 Génération de données de panel ...")

# Panel 1: Données équilibrées avec effets hétérogènes
entities_small = ['FR', 'DE', 'IT', 'ES']
df_panel = generate_panel_data(
    entities=entities_small,
    start_date='2015-01-01',
    periods=120,
    freq='MS',
    heterogeneous_effects=True,
    common_trend=True,
    entity_specific_seasonality=True,
    cross_sectional_correlation=0.4
)

# Suppression des Nan
df_panel.dropna(how='any', inplace=True)

print(f"✅ Génération de panels terminée:")
print(f"Caractéristiques des données générées : {df_panel.shape[0]} observations, {len(entities_small)} entités")

# Affichage des premières observations de chaque panel
print(f"\n📋 Aperçu des données:")
print(df_panel.head(10))

📊 Génération de données de panel ...
✅ Génération de panels terminée:
Caractéristiques des données générées : 476 observations, 4 entités

📋 Aperçu des données:
                      value  lag_value  trend  month
entity date                                         
FR     2015-02-01 -1.123838  -0.145034      1      2
       2015-03-01  0.807626  -1.123838      2      3
       2015-04-01 -0.796593   0.807626      3      4
       2015-05-01  0.806377  -0.796593      4      5
       2015-06-01  0.336501   0.806377      5      6
       2015-07-01  0.760570   0.336501      6      7
       2015-08-01  1.628513   0.760570      7      8
       2015-09-01  1.184729   1.628513      8      9
       2015-10-01  1.747917   1.184729      9     10
       2015-11-01 -1.066402   1.747917     10     11


## 2. Création d'un cadre de prévision à partir des éléments développpés dans `tsforecast`

In [4]:
# Séparation en X et y
y = df_panel['value'].copy()
X = df_panel.drop('value', axis=1)

# Initialisation de l'horizon de prédiction
horizon=2
# Initialisation du délai de publication
delays=1
# Application de l'horizon aux données afin d'aligner X et y à prévoir
# /!\ Créer une classe plus intelligente qui utilise la régularité de la série (sur données de panel et de séries temporelles) pour ajouter les dates manquantes aux extrémités de la période et ne pas perdre de données
X = X.shift(-horizon)

# Initialisation de la crossval pour l'ensemble des tests
cv = PanelOutOfSampleSplit(
    test_indices=['2024-01-01', '2024-02-01'], 
    test_size=1, 
    gap=horizon + delays
)
splits = list(cv.split(X, y))
# Extraction des indices d'entrainement et de test
train, test = splits[0]

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_indexing(X, train), _safe_indexing(y, train)
X_test = _safe_indexing(X, test)

## 3. Intégration des modèles de différents packages dans un cadre unifié, compatible avec les éléments développés dans `tsforecast`

### 3.1. Intégration des modèles `sklearn`

L'intégration de l'ensemble des estimateurs de `sklearn` dans le workflow se fait nativement, les observations `X` et `y` étant déjà alignées. La syntaxe est ainsi celle de `sklearn` :
- `.fit(X,y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... peuvent être utilisés de la même manière en respectant la syntaxe originale du package.

In [5]:
# Importation du modèle
from sklearn.linear_model import LinearRegression

# Initialisation du modèle
estimator=LinearRegression()

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_sklearn = estimator.predict(X_test)

y_pred_sklearn

array([-0.63153965,  3.54467872,  1.99541948,  7.22136797])

L'intégration des `Pipeline` de `sklearn.pipeline`, permettant de combiner des transformations opérées sur les données et l'entraînement d'un estimateur en fin de processus, se fait de la même manière en respectant la syntaxe originale.

In [6]:
# Initialisation de la pipeline
estimator = Pipeline([
    ('StandardScalerTransformer', StandardScaler()),
    ('RidgeEstimator', LinearRegression())
])

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_sklearn_pipeline = estimator.predict(X_test)

y_pred_sklearn_pipeline

array([-0.62366887,  3.5432445 ,  1.99743715,  7.21174175])

### 3.2. Intégration des modèles `xgboost`

L'API de `xgboost` est entièrement compatible avec celle de `sklearn`, aussi ses estimateurs s'intègrent nativement dans un workflow similaire. La syntaxe est ainsi celle de `sklearn` :
- `.fit(X,y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... de `sklearn` peuvent être utilisés avec ces modèles sans modification de la syntaxe.

In [7]:
# Initialisation du modèle
estimator=XGBRegressor()

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_split(estimator, X, y, train)
X_test, _ = _safe_split(estimator, X, y, test, train)
# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_xgboost = estimator.predict(X_test)

y_pred_xgboost

array([0.5864623, 2.6793466, 1.5268061, 5.0488734], dtype=float32)

 La combinaison de transformers avec les estimateurs de `xgboost` dans une `Pipeline` de `sklearn.pipeline` s'opère également sans difficulté.

In [8]:
# Importation du modèle
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# Initialisation de la pipeline
estimator = Pipeline([
    ('StandardScalerTransformer', StandardScaler()),
    ('XGBoostEstimator', XGBRegressor())
])

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_split(estimator, X, y, train)
X_test, _ = _safe_split(estimator, X, y, test, train)
# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_xgboost_pipeline = estimator.predict(X_test)

y_pred_xgboost_pipeline

array([0.5864623, 2.6793466, 1.5268061, 5.0488734], dtype=float32)

### 3.3. Intégration des modèles `tslearn`

`tslearn` reprend également l'API de `sklearn`, aussi ses estimateurs s'intègrent nativement dans un workflow similaire. La syntaxe est ainsi celle de `sklearn` :
- `.fit(X,y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... de `sklearn` peuvent être utilisés avec ces modèles sans modification de la syntaxe.

In [ ]:
# Importation du modèle
from tslearn.svm import TimeSeriesSVR

# Initialisation du modèle
estimator=TimeSeriesSVR()

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_split(estimator, X, y, train)
X_test, _ = _safe_split(estimator, X, y, test, train)
# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_tslearn = estimator.predict(X_test)

y_pred_tslearn

La combinaison de ces modèles avec des transformers de `sklearn` dans une `Pipeline` de `sklearn.Pipeline` s'opère de manière transparente

In [ ]:
# Importation du modèle
from sklearn.preprocessing import StandardScaler
from tslearn.svm import TimeSeriesSVR

# Initialisation de la pipeline
estimator = Pipeline([
    ('StandardScalerTransformer', StandardScaler()),
    ('SVREstimator', TimeSeriesSVR())
])

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_split(estimator, X, y, train)
X_test, _ = _safe_split(estimator, X, y, test, train)
# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_tslearn_pipeline = estimator.predict(X_test)

y_pred_tslearn_pipeline

### 3.4. Intégration des modèles `sktime`

#### 3.4.1. Intégration des modèles de `sktime.regression`, `sktime.classification` et `sktime.transformations`

Les modèles issus de ces modules suivent l'API de `sklearn`. Ils s'intègrent ainsi de la même manière dans le workflow avec la syntaxe :
- `.fit(X,y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... de `sklearn` peuvent être utilisés avec ces modèles sans modification de la syntaxe.

In [ ]:
# Importation du modèle
from sktime.regression.distance_based import KNeighborsTimeSeriesRegressor

# Initialisation du modèle
estimator=KNeighborsTimeSeriesRegressor()

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_split(estimator, X, y, train)
X_test, _ = _safe_split(estimator, X, y, test, train)
# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_sktime = estimator.predict(X_test)

y_pred_sktime

 La combinaison de transformers avec les estimateurs de `sktime.regression`, `sktime.classification` et `sktime.transformations` dans une `Pipeline` de `sklearn.pipeline` s'opère également sans difficulté.

In [ ]:
# Importation du modèle
from sktime.transformations.series.lag import Lag
from sklearn.impute import SimpleImputer
from sktime.regression.distance_based import KNeighborsTimeSeriesRegressor
from xgboost import XGBRegressor

# Initialisation de la pipeline
estimator = Pipeline([
    ('Lag2Transformer', Lag(2)),
    ('MeanImputerTransformer', SimpleImputer()),
    ('KNNEstimator', KNeighborsTimeSeriesRegressor())
])

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_split(estimator, X, y, train)
X_test, _ = _safe_split(estimator, X, y, test, train)
# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_sktime_pipeline = estimator.predict(X_test)

y_pred_sktime_pipeline

#### 3.4.2 Intégration des forecasters de `sktime.forecasting`

Les forecasters ont une syntaxe qui diffère légèrement de celle des modèles du package `sklearn` en ce que les 

##### 3.4.2.1 Utilisation des modèles de `sktime.forecasting`

##### 3.4.2.2. Utilisation des forecasters comme plateforme d'intégration des modèles de différents packages à partir de `make_reduction`

##### 3.4.2.3. Intégration des forecasters comme plateforme d'intégration des modèles de différents packages à partir de `YfromX`

### 3.5. Intégration des modèles `darts`

### 3.6. Intégration des modèles de `hierarchicalforecast`

### 3.7. Intégration des modèles de `opera`